**SALES ANALYSIS**

In [2]:
import pandas as pd
import os

**Merge the 12 months of sales data into a single CSV file**

In [5]:
files = [file for file in os.listdir('./sales_data') if file.endswith('.csv')]

all_months_data = pd.DataFrame()

for file in files:
    try:
        df = pd.read_csv(f'./sales_data/{file}', encoding='ISO-8859-1')
        all_months_data = pd.concat([all_months_data, df], ignore_index=True)
        print(f"Loaded: {file}")
    except UnicodeDecodeError as e:
        print(f"Unicode error in {file}: {e}")
    except Exception as e:
        print(f"Other error in {file}: {e}")
        
all_months_data.to_csv('all_data.csv', index=False)


Loaded: Sales_April_2019.csv
Loaded: Sales_August_2019.csv
Loaded: Sales_December_2019.csv
Other error in Sales_February_2019.csv: Error tokenizing data. C error: Expected 6 fields in line 8948, saw 8

Loaded: Sales_January_2019.csv
Loaded: Sales_July_2019.csv
Loaded: Sales_June_2019.csv
Loaded: Sales_March_2019.csv
Loaded: Sales_May_2019.csv
Loaded: Sales_November_2019.csv
Loaded: Sales_October_2019.csv
Loaded: Sales_September_2019.csv


**Read in updated dataframe**

In [7]:
all_data = pd.read_csv('all_data.csv')
all_data

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
0,176558,USB-C Charging Cable,2,11.95,04/19/19 08:46,"917 1st St, Dallas, TX 75001"
1,NaN,NaN,NaN,NaN,NaN,NaN
2,176559,Bose SoundSport Headphones,1,99.99,04/07/19 22:30,"682 Chestnut St, Boston, MA 02215"
3,176560,Google Phone,1,600,04/12/19 14:38,"669 Spruce St, Los Angeles, CA 90001"
4,176560,Wired Headphones,1,11.99,04/12/19 14:38,"669 Spruce St, Los Angeles, CA 90001"
...,...,...,...,...,...,...
174809,259353,AAA Batteries (4-pack),3,2.99,09/17/19 20:56,"840 Highland St, Los Angeles, CA 90001"
174810,259354,iPhone,1,700,09/01/19 16:00,"216 Dogwood St, San Francisco, CA 94016"
174811,259355,iPhone,1,700,09/23/19 07:39,"220 12th St, San Francisco, CA 94016"
174812,259356,34in Ultrawide Monitor,1,379.99,09/19/19 17:30,"511 Forest St, San Francisco, CA 94016"


**cleaning up the data**

In [9]:
all_data_cleaned = all_data.dropna(how='all')
## creating a month column
## Clean the Order Data column by removing any rows with invalid dates
all_data_cleaned = all_data_cleaned[all_data_cleaned['Order Date'].str.match(r'\d{2}/\d{2}/\d{2}.*').fillna(False)]

all_data_cleaned['Month'] = all_data_cleaned['Order Date'].str[0:2]

all_data_cleaned['Month'] = pd.to_numeric(all_data_cleaned['Month'], errors='coerce')
all_data_cleaned = all_data_cleaned.dropna(subset=['Month'])
all_data_cleaned['Month'] = all_data_cleaned['Month'].astype('int32')

all_data_cleaned

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,Month
0,176558,USB-C Charging Cable,2,11.95,04/19/19 08:46,"917 1st St, Dallas, TX 75001",4
2,176559,Bose SoundSport Headphones,1,99.99,04/07/19 22:30,"682 Chestnut St, Boston, MA 02215",4
3,176560,Google Phone,1,600,04/12/19 14:38,"669 Spruce St, Los Angeles, CA 90001",4
4,176560,Wired Headphones,1,11.99,04/12/19 14:38,"669 Spruce St, Los Angeles, CA 90001",4
5,176561,Wired Headphones,1,11.99,04/30/19 09:27,"333 8th St, Los Angeles, CA 90001",4
...,...,...,...,...,...,...,...
174809,259353,AAA Batteries (4-pack),3,2.99,09/17/19 20:56,"840 Highland St, Los Angeles, CA 90001",9
174810,259354,iPhone,1,700,09/01/19 16:00,"216 Dogwood St, San Francisco, CA 94016",9
174811,259355,iPhone,1,700,09/23/19 07:39,"220 12th St, San Francisco, CA 94016",9
174812,259356,34in Ultrawide Monitor,1,379.99,09/19/19 17:30,"511 Forest St, San Francisco, CA 94016",9


In [12]:
months_list = all_data_cleaned['Month'].unique()
for i in months_list:
    print(i, len(all_data_cleaned[all_data_cleaned['Month'] == i]))

4 18279
5 16566
8 11961
9 11621
12 24984
1 9709
2 6
7 14293
6 13554
3 15136
11 17573
10 20282


**adding a sales column**

In [14]:
all_data_cleaned['Sales'] = all_data_cleaned['Quantity Ordered'].astype('int32') * all_data_cleaned['Price Each'].astype('float64')

all_data_cleaned[['Product', 'Quantity Ordered', 'Price Each', 'Sales']]

,Product,Quantity Ordered,Price Each,Sales
0,USB-C Charging Cable,2,11.95,23.90
2,Bose SoundSport Headphones,1,99.99,99.99
3,Google Phone,1,600,600.00
4,Wired Headphones,1,11.99,11.99
5,Wired Headphones,1,11.99,11.99
...,...,...,...,...
174809,AAA Batteries (4-pack),3,2.99,8.97
174810,iPhone,1,700,700.00
174811,iPhone,1,700,700.00
174812,34in Ultrawide Monitor,1,379.99,379.99
